# 03b. Transformer Fine-Tuning on Kaggle (BioBERT / ClinicalBERT)
### Clinical Note-to-Billing Code Consistency Checker (FWA Triage)

> **Execution Environment**: Designed for Kaggle Notebooks with **GPU accelerator enabled** (NVIDIA Tesla P100 or T4).
>
> **Goal**: Fine-tune `dmis-lab/biobert-base-cased-v1.2` or `emilyalsentzer/Bio_ClinicalBERT` on clinical transcriptions to predict ICD/HCPCS code categories with high fidelity (~88–92% F1).

In [ ]:
# 1. Environment & GPU Verification
import torch
import sys
import os

print(f"Python: {sys.version}")
print(f"PyTorch version: {torch.__version__}")
gpu_available = torch.cuda.is_available()
print(f"CUDA Available: {gpu_available}")
if gpu_available:
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ WARNING: GPU not detected. Enable GPU in Kaggle settings: Settings -> Accelerator -> GPU P100.")

## 2. Install & Import Hugging Face Libraries

In [ ]:
!pip install -q transformers datasets accelerate evaluate scikit-learn

import pandas as pd
import numpy as np
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback
)
import evaluate

## 3. Load & Preprocess MTSamples Dataset

In [ ]:
# Kaggle path to dataset
KAGGLE_PATH = "/kaggle/input/medicaltranscriptions/mtsamples.csv"
LOCAL_PATH = "data/raw/mtsamples.csv"

data_file = KAGGLE_PATH if os.path.exists(KAGGLE_PATH) else LOCAL_PATH
df = pd.read_csv(data_file)
df = df.dropna(subset=['transcription', 'medical_specialty']).reset_index(drop=True)

# Map specialty to code category
SPECIALTY_MAP = {
    "Cardiovascular / Pulmonary": "circulatory",
    "Orthopedic": "musculoskeletal",
    "Gastroenterology": "digestive",
    "Neurology": "nervous",
    "Surgery": "surgical",
    "Urology": "genitourinary",
    "Dermatology": "skin_subcutaneous",
    "General Medicine": "general",
}
df['category'] = df['medical_specialty'].map(lambda s: SPECIALTY_MAP.get(s, "general"))

# Label Encoding
labels = sorted(df['category'].unique())
label2id = {l: i for i, l in enumerate(labels)}
id2label = {i: l for i, l in enumerate(labels)}
df['label'] = df['category'].map(label2id)

print(f"Total samples: {len(df):,}")
print(f"Classes ({len(labels)}): {labels}")

## 4. Tokenization & Dataset Preparation

In [ ]:
MODEL_NAME = "dmis-lab/biobert-base-cased-v1.2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Train/Val/Test Split (80/10/10)
from sklearn.model_selection import train_test_split
train_df, temp_df = train_test_split(df, test_size=0.2, stratify=df['label'], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['label'], random_state=42)

def tokenize_function(examples):
    return tokenizer(examples['transcription'], truncation=True, max_length=512)

train_ds = Dataset.from_pandas(train_df[['transcription', 'label']]).map(tokenize_function, batched=True)
val_ds = Dataset.from_pandas(val_df[['transcription', 'label']]).map(tokenize_function, batched=True)
test_ds = Dataset.from_pandas(test_df[['transcription', 'label']]).map(tokenize_function, batched=True)

## 5. Model Initialization & Trainer Setup

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id
)

f1_metric = evaluate.load("f1")
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=preds, references=labels)["accuracy"]
    f1_weighted = f1_metric.compute(predictions=preds, references=labels, average="weighted")["f1"]
    f1_macro = f1_metric.compute(predictions=preds, references=labels, average="macro")["f1"]
    return {"accuracy": acc, "f1_weighted": f1_weighted, "f1_macro": f1_macro}

training_args = TrainingArguments(
    output_dir="./biobert_clinical_classifier",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    fp16=torch.cuda.is_available(),
    load_best_model_at_end=True,
    metric_for_best_model="f1_weighted",
    logging_steps=50,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

## 6. Fine-Tuning & Evaluation

In [ ]:
# Run Training
trainer.train()

# Evaluate on held-out test set
test_results = trainer.evaluate(test_ds)
print("=== HELD-OUT TEST SET PERFORMANCE ===")
for k, v in test_results.items():
    print(f"  {k}: {v:.4f}")

## 7. Export Model Weights & Config for Local Inference

In [ ]:
# Save trained model to Kaggle output
output_export_dir = "./exported_biobert_model"
model.save_pretrained(output_export_dir)
tokenizer.save_pretrained(output_export_dir)

print(f"✅ Model successfully saved to {output_export_dir}")
print("Download these artifacts from Kaggle's Data Output tab to models/transformer/ locally.")